# **Лабораторная работа по криптографии №4<br>$\textit{Жуховицкий А. Д.}$ ( М8О-303Б-23 )<br>[Этот Jupyter на GitHub](google.com)**

## Разложение для числа a
Получим номер варианта:

In [1]:
from gostcrypto import gosthash
variant = gosthash.new("streebog512", data=bytes("Жуховицкий Александр Дмитриевич", "utf-8")).digest()[-1]
variant

244

Уложим все 256 вариантов задания в словарь, в файл я сложил весь контент документа кроме формулировки задания ЛР

In [2]:
data = dict()

with open("../data/keys.txt") as f:
    lines = f.readlines()
    for i in range(len(lines)):
        a = int(lines[i].split()[3])
        b = int(lines[i].split()[8])
        data[i] = (a, b)

Воспользуемся длинной арифметикой и скоростью готового пакета `sympy` для получения простых делителей числа `a` с primefactors и проверки простоты числа `b` при помощи вероятностной функции вшитой в `isprime`

In [3]:
from sympy import isprime, primefactors

Получим числа a и b для варианта

In [4]:
a, b = data[variant]
a, b

(431359146674424113353403426521803442507136093669191307915654746677889,
 32317006071311007300714876688669951960444102669715484032130345427524655138867890893197201411522913463688717960921898019494119559150490921095088152765270282947828371239440361392041508946042597047181039443674505095528015233509865144886442506266999758176624326351993070713707516845613527576730236488452166697896736816742630591634976243302604258831082455970293154694630056838840003543468664798684238540585672632939831791929841103687449568633583901796427884224462659744836174659197835258831287380228918309529697615076514735624671705249592910674204561095717070555004280786080783746492714481961909634076845110287175079460861)

In [5]:
divs = primefactors(a)
print(*divs)
a1, a2 = divs

18889465931479188520801 22835963083295358096932575511191922182699046689


Получили 2 простых делителя, произведение которых равно `a`

In [6]:
a1 * a2 == a

True

Число `a` разложено: `18889465931479188520801` * `22835963083295358096932575511191922182699046689` `=` `a`<br><br>
Проверим, хотя бы приблизительно, что `b` не является простым числом

In [7]:
isprime(b)

False

Действительно, почти точно число `b` не является простым.

## EDA (число b)

Проведём EDA по данным, которые нам доступны.<br>Числа b имеют схожее начало, а также схожую длину.

In [8]:
b

32317006071311007300714876688669951960444102669715484032130345427524655138867890893197201411522913463688717960921898019494119559150490921095088152765270282947828371239440361392041508946042597047181039443674505095528015233509865144886442506266999758176624326351993070713707516845613527576730236488452166697896736816742630591634976243302604258831082455970293154694630056838840003543468664798684238540585672632939831791929841103687449568633583901796427884224462659744836174659197835258831287380228918309529697615076514735624671705249592910674204561095717070555004280786080783746492714481961909634076845110287175079460861

In [9]:
len(bin(b)[2:])

2049

Числа не превышают $2^{2049}$<br>
Для разложения таких чисел не подойдут стандартные математические алгоритмы.<br>
Я попробовал использовать функции sympy для разложения, однако спустя довольно продолжительное время, результат не был получен. <br>
После небольшого исследования я узнал о ECM (Elliptic Curve Method) - алгоритме для разложения больших чисел, который может работать быстрее, чем классические алгоритмы, а также о GNFS (General Number Field Sieve) - алгоритме, который является самым эффективным для разложения больших чисел. <br>
ECM предназначен для разложения чисел, которые имеют относительно небольшие простые делители, в то время как GNFS предназначен для разложения чисел, которые не имеют таких делителей. <br>
Поскольку число `b` выглядит слишком большим для использования GNFS "в лоб", я решил попробовать использовать ECM для поиска какого-нибудь относительно небольшого простого делителя, который может помочь в разложении числа `b`.

In [53]:
%%bash
N='32317006071311007300714876688669951960444102669715484032130345427524655138867890893197201411522913463688717960921898019494119559150490921095088153957004883766528464610858080135791228755980166033045765418493393824525262986982969435184813482680389558802463395252303328128125469189367098585276037836712648167044379085982528495909695030081601422275311180111455197630737233947833293737848124261093855693104076713052007405746427759265741906753290321521176615398703044058404245694095881670037634441476249320049479478625889720062637890648093115392456100457097561743386344312627598903803022641629250476562404163521565125979557'

echo "$N" | ~/.opam/default/bin/ecm -v -c 200 3e8 2>&1 | head -n 7

GMP-ECM 7.0.5-dev [configured with GMP 6.3.0, --enable-assert] [ECM]
Tuned for default
Running on MacBook-Pro-zix.local
Input number is 32317006071311007300714876688669951960444102669715484032130345427524655138867890893197201411522913463688717960921898019494119559150490921095088153957004883766528464610858080135791228755980166033045765418493393824525262986982969435184813482680389558802463395252303328128125469189367098585276037836712648167044379085982528495909695030081601422275311180111455197630737233947833293737848124261093855693104076713052007405746427759265741906753290321521176615398703044058404245694095881670037634441476249320049479478625889720062637890648093115392456100457097561743386344312627598903803022641629250476562404163521565125979557 (617 digits)
Using MODMULN [mulredc:4, sqrredc:4]
Computing batch product (of 432807616 bits) of primes up to B1=300000000 took 6789ms
Using B1=300000000, B2=4022784085338, polynomial Dickson(30), sigma=1:3718551903


Однако ECM не дал никаких результатов.<br>
Также можно заметить, что числа `b` похожи на RSA-модули, которые состоят из произведения двух больших простых чисел. <br>
Числа `b[i]` имеют общий длинный префикс — все модули начинаются с `3231700607131100...`, что наводит на мысли о том, что для генерации чисел был использован "слабый" генератор случайных чисел при создании простых чисел `p` и `q`. Если два модуля  и  используют одно и то же простое число, то это немедленно раскрывает оба ключа.<br>
Заметим, что функция НОД (`gcd`), может помочь крайне быстро искать общие делители между числами `b[i]` благодаря реализации gcd с алгоритмом Евклида.<br>

## Batch GCD Attack
Попробуем провести `Batch GCD Attack` для `b` из моего варианта и `b[i]` из других вариантов, чтобы найти общий делитель, если он есть.

In [10]:
from math import gcd

gcd_num = 1
for i in range(256):
    gcd_num = gcd(b, data[i][1])
    if 1 < gcd_num < b:
        print(f"Found common divisor with b[{i}]: {gcd_num}")
        break

Found common divisor with b[108]: 13407807929942597099574024998205846127479365820592393377723561443721764030073546976801874298166903427690031858186486050853753882811946569946433649163251273


GCD атака прошла успешно!<br>
Найдём второй делитель для `b` и проверим, что оба делителя являются простыми числами.

In [11]:
b1 = int(gcd_num)
b2 = b // b1

In [12]:
b1 * b2 == b

True

In [13]:
isprime(b1), isprime(b2)

(True, True)

Найдены два простых и единственных делителя для числа `b` : `b1` и `b2`.

In [14]:
print(b1, b2)

13407807929942597099574024998205846127479365820592393377723561443721764030073546976801874298166903427690031858186486050853753882811946569946433649163251273 2410312426921032588580116606028314112912093247945688951359675039065257391591803200669085024107346049663448766280888004787862416978794958324969612987890774651455213339381625224770782077917681499676845543137387820057597345857904599109461387122099507964997815641342300677629473355281617428411794163967785870370368969109221591943054232011562758450080579587850900993714892283476646631181515063804873375182260506246992837898705971012525843324401232986857004761263707157


## Альтернативные варианты (полиномиальное разложение)
А также я узнал о том, что возможно разложить число `b` не используя остальные `b[i]`.<br>
Число `b` можно представить в виде полинома и значительно сократить варианты для перебора.<br>
Однако такой способ разложения будет работать только для чисел, которые имеют особенную структуру.<br>
Попробуем реализовать такое разложение.

In [15]:
from sympy import factor_list, symbols

def to_polynomial(n, base):
    powers = []

    cur_pow = 1
    while True:
        if n // cur_pow == 0:
            break

        powers.append(cur_pow)
        cur_pow *= base

    powers.reverse()

    result = []
    for p in powers:
        result.append(n // p)
        n = n % p

    return result

Проведём факторизацию числа через его полиномиальное представление в некотором основании $2^{i}$.<br>
В коде число  сначала записывается в системе счисления с основанием , затем эта запись рассматривается как многочлен.<br>
Будем перебирать основания начиная с $2^{2048}$, так как число `b` не превышает $2^{2049}$ как было выяснено ранее.

In [16]:
from IPython.display import display, Math
from sympy import latex

x = symbols('x')
i = 2048
c = 1
power = 2 ** i
factors = []

while i >= 4:
    power = 2 ** i
    res = to_polynomial(b, power)

    non_zero_items = [(idx, coef) for idx, coef in enumerate(res) if coef != 0]

    if len(non_zero_items) % 2 == 0:
        equation = sum(coef * x ** (len(res) - idx - 1) for idx, coef in non_zero_items)

        c, factors = factor_list(equation)
        if len(factors) < 2:
            i //= 2
            continue

        equation_tex = latex(equation, mul_symbol=r' \ast ')

        display(Math(rf"\text{{Let }} x = 2^{{{i}}},\ \text{{then }} b = {equation_tex}"))
        display(Math(
            rf"\text{{Factorization: }} {c} \cdot "
            + r"\left("
            + r" \cdot ".join(
                [rf"\left({latex(num)}\right)^{{{num_power}}}" for num, num_power in factors]
            )
            + r"\right)"
        ))
        break
    i //= 2

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [17]:
b1_poly = c * int(factors[0][0].subs(x, power)) ** int(factors[0][1])
b2_poly = int(factors[1][0].subs(x, power)) ** int(factors[1][1])
print(b1_poly, b2_poly, sep="\n")


13407807929942597099574024998205846127479365820592393377723561443721764030073546976801874298166903427690031858186486050853753882811946569946433649163251273
2410312426921032588580116606028314112912093247945688951359675039065257391591803200669085024107346049663448766280888004787862416978794958324969612987890774651455213339381625224770782077917681499676845543137387820057597345857904599109461387122099507964997815641342300677629473355281617428411794163967785870370368969109221591943054232011562758450080579587850900993714892283476646631181515063804873375182260506246992837898705971012525843324401232986857004761263707157


Проверим, что разложение числа `b` в виде полинома действительно даёт нам те же простые делители, что и GCD атака.

In [18]:
b1 == b1_poly and b2 == b2_poly

True